# Cloud Run Inference Service Setup

This notebook documents the Google Cloud Run inference service used by the cross-platform machine-learning experiments.

The cloud service is documented separately from any particular FPGA or Unity client. The first implementation is MNIST:

```text
Service:  mnist-cloud-inference
Project:  mnist-cloud-fpga
Region:   us-east1
Endpoint: /predict
```

Later dataset/model changes can be added to this notebook.


## 1. Cloud Run service

The MNIST service was deployed in `us-east1` at:

```text
https://mnist-cloud-inference-221036165238.us-east1.run.app
```

Inference requests were sent to:

```text
POST /predict
```

The service hostname should be used rather than a fixed frontend IP address.


## 2. Source files

The MNIST deployment used:

```text
main.py
network2.py
requirements.txt
WeigntsAndBiases.txt
```

The deployed `main.py` imports Flask, NumPy, and `network2`, creates the Flask application, and loads the trained floating-point network with:

```python
net = network2.load("WeigntsAndBiases.txt")
```

The service also provides a root health endpoint that returns:

```text
MNIST cloud inference service is running
```

The `/predict` endpoint performs inference.


## 3. Python dependencies

`requirements.txt` used:

```text
Flask==3.0.3
gunicorn==23.0.0
numpy==1.26.4
```


## 4. MNIST request path

The `/predict` endpoint accepts a raw POST request body containing exactly **784 bytes**.

If the request body is not exactly 784 bytes, the service returns HTTP 400 with a JSON error indicating the expected and received byte counts.

For a valid request, the server:

```text
784 raw bytes
    ↓
np.frombuffer(..., dtype=np.uint8)
    ↓
convert to float64
    ↓
divide by 128.0
    ↓
reshape to 784 × 1
    ↓
net.feedforward(x)
    ↓
np.argmax(output)
    ↓
JSON prediction
```

The normalization by `128.0` matches the 8-bit fixed-point input representation used by the FPGA path.

The response format is:

```json
{"prediction": 7}
```

The Flask application binds to `0.0.0.0` and uses the `PORT` environment variable, defaulting to port 8080 when run directly.


## 5. Service revision and deployment identity

The exported request logs identify:

```text
service_name:       mnist-cloud-inference
configuration_name: mnist-cloud-inference
project_id:         mnist-cloud-fpga
location:           us-east1
```

One recorded revision was:

```text
mnist-cloud-inference-00002-plx
```

Revision identifiers change when the service is redeployed.


## 6. Access configuration

The Cloud Run Security view showed IAM authentication controls enabled. The service was also publicly accessible because `allUsers` had permission on the service.

That allowed the experimental FPGA and Unity clients to call the endpoint without Google Cloud identity credentials.

This was an experimental deployment choice rather than a production security recommendation.


## 7. Encryption

The service used a Google-managed encryption key. No customer-managed Cloud KMS key was configured.


## 8. Observability

Cloud Run observability was used to verify:

- request count;
- request latency;
- end-to-end request latency;
- latency breakdown;
- scaling activity;
- errors; and
- billing/cost information.


## 9. Exported request logs

The branch includes:

```text
Cloud_MNIST_logs.json
```

The exported logs record successful `POST` requests to `/predict` with HTTP status `200`. They also include request size, response size, Cloud Run latency, revision, timestamps, protocol, and client user agent.

The logs include requests from Unity identified by the `UnityPlayer` user agent.

Cloud Run request latency is a server/platform-side measurement and should not be treated as equivalent to client-side end-to-end latency.


In [ ]:
import json
from pathlib import Path

log_path = Path("Cloud_MNIST_logs.json")

if log_path.exists():
    logs = json.loads(log_path.read_text(encoding="utf-8"))
    print("Log entries:", len(logs))
    if logs:
        req = logs[0].get("httpRequest", {})
        labels = logs[0].get("resource", {}).get("labels", {})
        print("Method:", req.get("requestMethod"))
        print("URL:", req.get("requestUrl"))
        print("Status:", req.get("status"))
        print("Protocol:", req.get("protocol"))
        print("Cloud Run latency:", req.get("latency"))
        print("Service:", labels.get("service_name"))
        print("Region:", labels.get("location"))
else:
    print("Place Cloud_MNIST_logs.json in the same directory as this notebook.")


## 10. Screenshots

The screenshots in this branch document the Cloud Run service, source files, security settings, observability, scaling, and errors.


### Service and scaling

![Cloud Run service and scaling](<Cloud_MNIST_Scaling.png>)


### Source and dependencies

![Cloud Run source files and requirements](<Cloud_MNIST_Sources.png>)


### Security and encryption

![Cloud Run security configuration](<Cloud_MNIST_Security.png>)


### Observability

![Cloud Run observability](<Cloud_MNIST_Metrics.png>)


### Error reporting

![Cloud Run error dashboard](<Cloud_MNIST_Errors.png>)


## 11. Separation from FPGA and Unity clients

This branch documents the cloud service only.

FPGA branches document Ethernet, lwIP, DNS, TCP, TLS, FPGA/PS mailbox communication, request transmission, returned predictions, and FPGA-side measurements.

Unity branches document UnityWebRequest, input preparation, response handling, and PC/Quest measurements.

This keeps the Cloud Run service common while each client documents its own implementation and timing boundary.


## 12. Dataset/model update template

For each later dataset, add:

```text
Dataset:
Cloud Run service name:
Region:
Model file:
Model architecture:
Input dimensions:
Input data type:
Input normalization:
Request body format:
Response format:
main.py changes:
Model-loading changes:
requirements.txt changes:
Endpoint changes:
Cloud Run revision:
Deployment date:
Validation performed:
```


## 13. MNIST deployment summary

```text
client
   ↓
HTTPS POST /predict
   ↓
Cloud Run
   ↓
Flask application
   ↓
784 input values
   ↓
784–30–20–10 neural network
   ↓
predicted digit
   ↓
JSON response
```

The same remote inference service could be used by multiple client platforms while each platform measured its own communication and execution behavior separately.


## 14. Branch purpose

This branch provides a record of the cloud inference environment used by the dissertation experiments.

As additional datasets and models are deployed, this notebook should document changes to the Cloud Run application and model interface while FPGA- and Unity-specific implementation details remain in their own branches.
